In [ ]:
# This script uses class definition to store, access and compute GP Observables in an organized manner

import GPKoopman as gpk
import torch
import numpy as np
import matplotlib.pyplot as plt
# import matplotlib as mpl
# from mpl_toolkits.mplot3d import Axes3D
# from IPython.display import display, clear_output
import math
from matplotlib.patches import Ellipse
import datetime
# import itertools
from sklearn.cluster import KMeans

print('Imported PyTorch version:', torch.__version__)
print('Imported NumPy version:', np.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

### Plotting Functions

In [ ]:
def plot_time_series_with_bounds(time, Xhat, Xcvhat, SimData, idx, N, system_name, title_suffix, sim_offset=0):
    """
    Plots time series for each state with uncertainty bounds.
    
    Parameters:
        time        : 1D array of time points.
        Xhat        : Array of model state estimates.
        Xcvhat      : Array of state covariance estimates.
        SimData     : Array of true system states.
        idx         : Index of the trajectory to plot.
        N           : Number of time steps to plot.
        system_name : Name of the system (for labeling).
        title_suffix: Suffix for the plot title.
        sim_offset  : Offset for SimData indexing (e.g., nTrain for test trajectories).
    """
    n_states = Xhat.shape[1]
    fig, axes = plt.subplots(n_states, 1, figsize=(6, 5))
    
    # If there is only one state, wrap the Axes object in a list for uniform handling.
    if n_states == 1:
        axes = [axes]
    
    fig.suptitle(f'{system_name}: {title_suffix}')
    
    for i in range(n_states):
        lower_bound = Xhat[idx, i, :] - (Xcvhat[idx, i, i, :] ** 0.5)
        upper_bound = Xhat[idx, i, :] + (Xcvhat[idx, i, i, :] ** 0.5)
        axes[i].fill_between(time, lower_bound, upper_bound, alpha=0.16, color='blue')
        axes[i].plot(time, Xhat[idx, i, :], label='iGPK', color='blue')
        axes[i].plot(time, SimData[sim_offset + idx, i, :N], label='NL', linestyle='--', color='red')
        axes[i].set_ylabel(f'State X{i+1}')
        axes[i].legend()
        axes[i].grid()
        if i < n_states - 1:
            axes[i].set_xticklabels([])  # Hide x-axis labels for intermediate subplots
    
    axes[-1].set_xlabel('Time [s]')
    plt.tight_layout()
    plt.show()

def plot_phase(Xhat, SimData, ICset, idx, N, system_name, title_suffix, sim_offset=0):
    """
    Plots the phase trajectory comparing the model prediction to the true (nonlinear) system.
    Automatically switches to a 3D plot if the system has 3 state dimensions.

    Parameters:
        Xhat       : Array of model state estimates of shape (trajectories, states, time_steps).
        SimData    : Array of true system states.
        ICset      : Array of initial conditions.
        idx        : Index of the trajectory to plot.
        N          : Number of time steps to plot.
        system_name: Name of the system (for labeling).
        title      : Title for the plot.
        sim_offset : Offset for SimData indexing (e.g., nTrain for test trajectories).
    """
    n_states = Xhat.shape[1]
    
    if n_states == 3:
        # Create a 3D plot for 3-state systems
        fig = plt.figure(figsize=(6, 4.5))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot(Xhat[idx, 0, :], Xhat[idx, 1, :], Xhat[idx, 2, :], label='iGPK')
        ax.plot(SimData[sim_offset + idx, 0, :],
                SimData[sim_offset + idx, 1, :],
                SimData[sim_offset + idx, 2, :], label='Nonlinear', linestyle='--')
        ax.scatter(ICset[0, idx], ICset[1, idx], ICset[2, idx], label='IC', marker='o')
        ax.set_title(f'{system_name}: {title_suffix}')
        ax.set_xlabel("X1")
        ax.set_ylabel("X2")
        ax.set_zlabel("X3")
        ax.legend()
        plt.show()
    else:
        # Default to a 2D phase plot (using the first two state dimensions)
        plt.figure(figsize=(6, 4.5))
        plt.plot(Xhat[idx, 0, :], Xhat[idx, 1, :], label='iGPK')
        plt.plot(SimData[sim_offset + idx, 0, :],
                 SimData[sim_offset + idx, 1, :],
                 label='Nonlinear', linestyle='--')
        plt.plot(ICset[0, idx], ICset[1, idx], label='IC', marker='o')
        plt.title(f'{system_name}: {title_suffix}')
        plt.xlabel("X1")
        plt.ylabel("X2")
        plt.legend()
        plt.grid()
        plt.show()

def plot_phase_w_bounds(Xhat, SimData, ICset, idx, N, system_name, title_suffix, sim_offset=0, Xcvhat=None, ellipse_interval=10):
    """
    Plots the phase trajectory comparing the model prediction to the true system.
    Automatically switches to a 3D plot if the system has 3 state dimensions.
    For 2D systems, if Xcvhat is provided, error ellipses (shaded uncertainty regions)
    are added along the trajectory.

    Parameters:
        Xhat          : Array of model state estimates, shape (trajectories, states, time_steps).
        SimData       : Array of true system states.
        ICset         : Array of initial conditions.
        idx           : Index of the trajectory to plot.
        N             : Number of time steps to plot.
        system_name   : Name of the system (for labeling).
        title         : Title for the plot.
        sim_offset    : Offset for SimData indexing (e.g., nTrain for test trajectories).
        Xcvhat        : (Optional) Array of state covariance estimates with shape 
                        (trajectories, states, states, time_steps).
        ellipse_interval: Interval (in time steps) at which to plot error ellipses.
    """
    n_states = Xhat.shape[1]
    
    if n_states == 3:   # Create a 3D phase plot
        fig = plt.figure(figsize=(6, 4.5))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot(Xhat[idx, 0, :], Xhat[idx, 1, :], Xhat[idx, 2, :], label='iGPK')
        ax.plot(SimData[sim_offset + idx, 0, :N],
                SimData[sim_offset + idx, 1, :N],
                SimData[sim_offset + idx, 2, :N],
                label='Nonlinear', linestyle='--')
        ax.scatter(ICset[0, idx], ICset[1, idx], ICset[2, idx], label='IC', marker='o')
        ax.set_title(f'{system_name}: {title_suffix}')
        ax.set_xlabel("X1")
        ax.set_ylabel("X2")
        ax.set_zlabel("X3")
        ax.legend()
        plt.show()

    elif n_states == 2: # 2-D phase plot
        # Create a 2D phase plot
        plt.figure(figsize=(6, 4.5))
        plt.plot(Xhat[idx, 0, :], Xhat[idx, 1, :], label='iGPK')
        plt.plot(SimData[sim_offset + idx, 0, :N],
                 SimData[sim_offset + idx, 1, :N],
                 label='Nonlinear', linestyle='--')
        plt.plot(ICset[0, idx], ICset[1, idx], label='IC', marker='o')
        
        # If covariance data is provided, overlay error ellipses as uncertainty regions.
        if Xcvhat is not None:  # shaded elipses for 1-sigma bound
            for t in range(0, N, ellipse_interval):
                center = (Xhat[idx, 0, t], Xhat[idx, 1, t])
                # Extract the 2x2 covariance matrix for the first two states
                cov = Xcvhat[idx, 0:2, 0:2, t]
                # Convert to NumPy array if it's a torch tensor
                if hasattr(cov, "cpu"):
                    cov = cov.cpu().numpy()
                # Compute eigenvalues and eigenvectors of the covariance matrix
                vals, vecs = np.linalg.eig(cov)
                order = vals.argsort()[::-1]
                vals = vals[order]
                vecs = vecs[:, order]
                # Compute the angle of the ellipse (in degrees)
                angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
                # For a 1-sigma ellipse, the semi-axis lengths are the square roots of the eigenvalues.
                width, height = 2 * np.sqrt(vals)
                ellipse = Ellipse(xy=center, width=width, height=height, angle=angle,
                                  edgecolor='blue', facecolor='blue', alpha=0.1)
                plt.gca().add_patch(ellipse)
        
        if Xcvhat is None:  # Plot Title
            plt.title(f'{system_name}: {title_suffix}')
        else:   # Plot Title for 1 sigma bounds
            plt.title(f'{system_name}: {title_suffix} with 1 $\\sigma$ bound')

        plt.xlabel("X1")
        plt.ylabel("X2")
        plt.legend()
        plt.grid()
        plt.show()
        
    else:   # Dimension mismatch
        raise ValueError('Size of Xhat in dimension 1 has to be 2 or 3')

def plot_predicted_sd_error(XcvhatTest, SimData, XhatTest, idx, N, nTrain, trajectory_label):
    """
    Plots the predicted standard deviation (SD) and absolute error for a given test trajectory.
    
    Parameters:
        XcvhatTest      : Tensor of covariance estimates, shape (trajectories, states, states, time_steps).
        SimData         : Tensor of true system states, shape (num_trajectories, states, time_steps).
        XhatTest        : Tensor of predicted state estimates, shape (trajectories, states, time_steps).
        idx             : Index of the trajectory to plot.
        N               : Number of time steps to plot.
        nTrain          : Offset for test trajectories in SimData.
        trajectory_label: String label for the trajectory (e.g., "Worst Test", "Best Test").
    """
    n = XhatTest.shape[1]
    fig, axes = plt.subplots(n, 1, figsize=(6, 5))
    
    # Protection for 1D systems: ensure axes is always iterable.
    if n == 1:
        axes = [axes]
    
    fig.suptitle(f'Predicted SD & |Error| for {trajectory_label} Trajectory ({idx})')
    
    for i in range(n):
        # Compute standard deviation from covariance (for a 1-sigma bound)
        sigma = (torch.abs(XcvhatTest[idx, i, i, :N-1]) ** 0.5)
        # Compute absolute error between true and predicted values
        error = torch.abs(SimData[nTrain+idx, i, :N-1] - XhatTest[idx, i, :N-1])
        
        # Optionally, convert tensors to NumPy for plotting (if needed)
        sigma = sigma.detach().cpu().numpy()
        error = error.detach().cpu().numpy()
        
        axes[i].plot(sigma, label='$\\sigma_X$', color='blue')
        axes[i].plot(error, label='$\\epsilon_X$', color='red', linestyle='--')
        axes[i].set_ylabel(f'X{i+1}')
        axes[i].grid()
        axes[i].legend()
        if i < n - 1:
            axes[i].set_xticklabels([])  # Hide x-axis labels for all but the last subplot

    axes[-1].set_xlabel('Time Step')
    plt.tight_layout()
    plt.show()

### Utility Functions

In [ ]:
def check_pd(matrix: torch.Tensor):
    """
    Checks if a given 2D PyTorch tensor is positive definite (PD) or 
    positive semi-definite (PSD) or neither.
    
    Args:
        matrix (torch.Tensor): A 2D square PyTorch tensor.

    Returns:
        str: "Positive Definite", "Positive Semi-Definite", or "Neither"
    """
    if not (matrix.dim() == 2 and matrix.shape[0] == matrix.shape[1]):
        raise ValueError("Input must be a square matrix")

    try:
        # Attempt Cholesky decomposition for positive definiteness
        torch.linalg.cholesky(matrix)
        return "Positive Definite"
    except RuntimeError:
        # Compute eigenvalues for semi-definiteness check
        eigenvalues = torch.linalg.eigvals(matrix)
        if torch.all(eigenvalues >= 0):
            return "Positive Semi-Definite"
        else:
            return "Neither"

def get_kmeans(data, num_centers=1):
    """
    Compute K-Means clustering for the given data and return cluster centroids.

    Args:
        data (torch.Tensor or np.ndarray): Data of shape (dimension, samples).
        num_centers (int): Number of cluster centers (clusters) to compute.
    
    Returns:
        torch.Tensor: Cluster centroids of shape (dimension, num_centers).
    """
    # Convert data to NumPy array if necessary
    if isinstance(data, torch.Tensor):
        data_np = data.detach().cpu().numpy()
    else:
        data_np = np.array(data)
    
    # Transpose data to shape (samples, dimension) for scikit-learn
    data_np = data_np.T
    
    # Perform k-means clustering
    kmeans = KMeans(n_clusters=num_centers, random_state=0).fit(data_np)
    
    # Get centroids; shape will be (num_centers, dimension)
    centroids_np = kmeans.cluster_centers_
    
    # Convert centroids to a torch tensor and transpose to shape (dimension, num_centers)
    centroids = torch.from_numpy(centroids_np.T).float()
    
    return centroids

### Cost Function Definition

In [ ]:
# Multi-Trajectory Cost Function Definition

def get_cost_grid(Z, X, Xplus, Xtrain, manager, nT=1, lambda1=1.0, lambda2=1.0, lambda3=1.0):
    """
    Computes the cost function as defined in the Word doc, with different GP Kernel hyperparameters for
    each observable.

    Args:
        Z: Tensor of shape (r**n, p), decision variable, required grad
        X: Tensor of shape (n, nT*N), dataset of N steps from all trajectories
        Xall: Tensor of shape (n, nT*(N+1)), complete training dataset
        Xtrain: Tensor of shape (n, r**n), flattened set of gridpoints for training GPOs
        manager: Object of class GPObservablesManager (manager for all Gaussian Process based Observable functions)
        nT: float, number of trajectories in training dataset
        lambda1: float, Weighting for prediction error minimization term
        lambda2: float, Weightining for Reconstruction Error penalty term
    """
    
    N = (X.shape[1])//nT    # Number of time steps in each trajectory
    p = Z.shape[1]          # Number of Observables
    l = Z.shape[0]//nT      # Decision Horizon
    n = X.shape[0]          # Dimensionality of original system

    for i in range(p):
        manager.train_observable(i, Xtrain, Z[:,i])

    # For current definition of GPOs
    # Training: Xtrain = dimensions x samples
    # Training: Ytrain = samples x (dimensions=1)
    # Prediction: Xquery = dimensions x num-query = Input
    # Prediction: Yquery = num-query x (dimensions=1) = Output

    # Lifting X and Xplus to higher dimension using trained GPOs
    M = torch.empty((p,N*nT))
    Mplus = torch.empty((p,N*nT))
    
    #Mall = torch.empty((p,(N+1)*nT))
    for i in range(p):
        M[i,:] = torch.transpose(manager.predict_mean(i, X), dim0=0, dim1=-1)
        Mplus[i,:] = torch.transpose(manager.predict_mean(i, Xplus), dim0=0, dim1=-1)
    
    # Compute C(z) and A(z)
    Mfull = torch.vstack((X, M))
    Mplusfull = torch.vstack((Xplus, Mplus))

    M_pinv = torch.linalg.pinv(Mfull)
    #Cz = X @ M_pinv
    Az = Mplusfull @ M_pinv
    C = torch.zeros((n,n+p))
    for i in range(n):
        C[i,i] = 1.
    
    # Cost term 1: Multi-Trajectory Prediction Error Minimization
    NormPEM = 0.0
    for j in range(nT):
        TrajPEM = 0.0
        for k in range(N - 1):
            pred_error = X[:, j*N + (k+1)] - C @ (torch.linalg.matrix_power(Az,k+1)) @ Mfull[:,j*N]   # multi-step at X (with Cz)
            TrajPEM += torch.norm(pred_error)
        NormPEM += TrajPEM
    
    # Linearity Enforcement
    NormLEP = 0.0
    for j in range(nT):
        TrajLEP = 0.0
        for k in range(l-1):
            Zk = torch.vstack([X[:, j*N + k].view(n,1), torch.transpose(Z[j*l + k, :], dim0=0, dim1=-1).view(p,1)])
            Zkplus = torch.vstack([X[:, j*N + k + 1].view(n,1), torch.transpose(Z[j*l + k + 1, :], dim0=0, dim1=-1).view(p,1)])
            lin_error = Zkplus - Az @ Zk
            TrajLEP += torch.norm(lin_error)
        NormLEP += TrajLEP
    
    # Weighted sum of terms
    cost = (lambda1 * NormPEM / (N * nT)) + (lambda2 * NormLEP / (l * nT))
    return cost

def get_cost_AC(Z, X, Xplus, Xtrain, manager, nT=1, lambda1=1.0, lambda2=1.0, lambda3=1.0):
    """
    Computes the cost function using a single differentiable GP forward pass per observable,
    merging the training and prediction steps by passing Z[:, i] directly to the forward method.
    
    Args:
        Z: Tensor of shape (r**n, p), decision variable (requires grad).
        X: Tensor of shape (n, nT*N), dataset of N steps per trajectory.
        Xplus: Tensor of shape (n, nT*N), time-shifted dataset.
        Xtrain: Tensor of shape (n, r**n), gridpoints for training.
        manager: GPObservablesManager.
        nT: Number of trajectories.
        lambda1: Weighting for NLPD.
        lambda2: Weighting for linearity enforcement.
        lambda3: Weighting for prediction error minimization.
    """
    N = X.shape[1] // nT    # Number of time steps per trajectory
    p = Z.shape[1]          # Number of observables
    l = Z.shape[0] // nT    # Decision horizon
    n = X.shape[0]          # State dimension

    # For each observable, call forward once on the full dataset X (and Xplus)
    M = torch.empty((p, N * nT), device=X.device)
    cov_all = [None] * p  # store full covariance matrices for X
    Mplus = torch.empty((p, N * nT), device=X.device)
    # cov_all_plus = [None] * p  # store full covariance matrices for Xplus

    for i in range(p):
        mean_i, cov_i = manager.observables[i].forward(X, Z[:, i])
        M[i, :] = torch.transpose(mean_i, 0, -1)
        cov_all[i] = cov_i

        mean_plus_i, _ = manager.observables[i].forward(Xplus, Z[:, i])
        Mplus[i, :] = torch.transpose(mean_plus_i, 0, -1)

    # Compute the pseudo-inverse lifting operator and the corresponding matrices Cz and Az.
    M_pinv = torch.linalg.pinv(M)
    Cz = X @ M_pinv
    Az = Mplus @ M_pinv

    # Cost Term 1: Negative Log Predictive Density (NLPD)
    NormNLPD = 0.0
    if not math.isclose(lambda1, 0):
        for j in range(nT):
            TrajNLPD = 0.0
            # Define the number of time steps for NLPD computation:
            num_steps = N - 2 - l
            vz_k = torch.empty((p, num_steps), device=X.device)
            for i in range(p):
                # For trajectory j, determine the indices for the NLPD slice.
                start = j * N + l + 1
                end = (j + 1) * N - 1  # end is exclusive
                # Extract the covariance for the slice from the precomputed full covariance.
                cov_sub = cov_all[i][start:end, start:end]
                # Get the diagonal elements (predictive variances) and clamp them.
                vz_k[i, :] = torch.clamp(torch.diag(cov_sub), min=1e-3)
            for k in range(num_steps):
                vx_next = torch.abs(torch.diag(Cz @ Az @ torch.diag(vz_k[:, k]) @ Az.T @ Cz.T).view(n, 1))
                error_term = ((X[:, j * N + l + 1 + k + 1] - Cz @ Az @ M[:, j * N + l + 1 + k]) ** 2) / vx_next
                log_term = torch.log(vx_next)
                TrajNLPD += torch.sum(error_term + log_term)
            NormNLPD += TrajNLPD

    # Cost Term 2: Linearity Enforcement
    NormLEP = 0.0
    if not math.isclose(lambda2, 0):
        for j in range(nT):
            TrajLEP = 0.0
            for k in range(l - 1):
                lin_error = torch.transpose(Z[j * l + k + 1, :], 0, -1) - Az @ torch.transpose(Z[j * l + k, :], 0, -1)
                TrajLEP += torch.norm(lin_error)
            NormLEP += TrajLEP

    # Cost Term 3: Prediction Error Minimization
    NormPEM = 0.0
    if not math.isclose(lambda3, 0):
        for j in range(nT):
            TrajPEM = 0.0
            for k in range(N - 1):
                pred_error = X[:, j * N + (k + 1)] - Cz @ (torch.linalg.matrix_power(Az, k + 1)) @ M[:, j * N]
                TrajPEM += torch.norm(pred_error)
            NormPEM += TrajPEM

    cost = (lambda1 * NormNLPD / ((N - l) * nT)) + (lambda2 * NormLEP / (l * nT)) + (lambda3 * NormPEM / (N * nT))
    return cost


# Execution Cells

## Data Loading

In [ ]:
# Allowed system names -
# "Unforced Duffing" | "Unforced Duffing_right" | "van der Pol" | "Simple Pendulum"
# "Lorenz" | "Lotka Volterra" | "Piecewise Linear"

system_name = 'Unforced Duffing'
data = torch.load(f"Data/DataAuto_{system_name}.pt", weights_only=True)

SimData = data["trajectories"] # Shape: (num_trajectories, state_dim, num_steps)
ts = data["sample_time"]
num_trajectories = data["num_trajectories"]
N = data["num_steps"]

nTrain, nTest = math.floor(num_trajectories * 0.3), math.floor(num_trajectories * 0.2)

# Clip Training Data steps
SimData = SimData[:,:,:101]
N = 100

## Optimization Section

### Optimization Setup

In [ ]:
SimData = SimData.float()

# Original State Dim | Lifted State Dim | Decision Horizon | Resolution
n, p, l, r = SimData.shape[1], 10, 20, 10

Xall = torch.cat([SimData[j, :, :] for j in range(nTrain)], dim=1)      # Concatenated total matrix
X = torch.cat([SimData[j, :, 0:N] for j in range(nTrain)], dim=1)       # Concatenated Data matrix
Xplus = torch.cat([SimData[j, :, 1:] for j in range(nTrain)], dim=1)    # Time-shifted Data matrix

ICsetTrain = torch.cat([SimData[j, :, 0].view(n,1) for j in range(nTrain)], dim=1)    # Random IC set for training
ICsetTest = torch.cat([SimData[j, :, 0].view(n,1) for j in range(nTrain, nTrain + nTest)], dim=1)  # Random IC set for testing

# Options: 'Horizon' | 'Grid' | 'K-Means'
trainMethod = 'K-Means'

# Initialize GP training-grid and decision variables
if trainMethod == 'Horizon':
    Xtrain = torch.cat([X[:,j*N:j*N+l] for j in range(nTrain)],dim=1)
    #Z = torch.rand(Xtrain.shape[1], p, requires_grad=True)
    Z = torch.nn.Parameter(torch.rand(Xtrain.shape[1], p))
    ObsManager = gpk.GPObservablesManager()
    for i in range(p):
        ObsManager.add_observable(index=i, d=n, ns=l*nTrain, kernel_types=['Gaussian'], combination='sum',noise=1e-6, m=500)
    for i in range(p):
        ObsManager.train_observable(i, Xtrain, Z[:, i])
    ObsManager.set_random_hyperparameters(scale=[1., 0.01, None])
    print('Observable Hyperparameters have been randomized:')
    ObsManager.print_parameters()

elif trainMethod == 'K-Means':
    Xtrain = torch.cat([X[:,j*N:j*N+l] for j in range(nTrain)],dim=1)
    Z = torch.nn.Parameter(torch.rand(Xtrain.shape[1], p))
    ObsManager = gpk.GPObservablesManager()
    centroids = get_kmeans(X, num_centers=p)
    for i in range(p):
        ObsManager.add_observable(index=i, d=n, ns=l*nTrain, kernel_types=['GibbsExpAttractor'], combination='sum',noise=1e-6, m=500)
    
    mu_centroids = [centroids[:, i:i+1] for i in range(centroids.shape[1])]
    ObsManager.set_parameters(mu_list=mu_centroids)
    for i in range(p):
        ObsManager.train_observable(i, Xtrain, Z[:, i])
    ObsManager.set_random_hyperparameters(scale=[1., 0.01, None])
    print('Observable Hyperparameters have been randomized:')
    ObsManager.print_parameters()

elif trainMethod == 'Grid':
    # GPO Training Grid
    gridpoints0 = torch.linspace(-4., 4., steps=r)
    gridpoints1 = torch.linspace(-4., 4., steps=r)
    grid0, grid1 = torch.meshgrid(gridpoints0, gridpoints1, indexing='xy')
    Xtrain = torch.stack([torch.flatten(grid0), torch.flatten(grid1)])
    Z = torch.rand(r**n, p, requires_grad=True)
    ObsManager = gpk.GPObservablesManager()
    for i in range(p):
        ObsManager.add_observable(index=i, d=n, ns=r**n, kernel_types=['Gaussian', 'ExplicitAttractor'], combination='product', noise=1e-6)

else:
    raise ValueError(f'Unrecognized GP Training method {trainMethod}')

In [ ]:
if trainMethod == 'K-Means':
    for i in range(p):
        plt.plot(centroids[0,i].cpu(), centroids[1,i].cpu(), marker='o')
    plt.grid()
    plt.title('Centroids of K-Means Clusters'), plt.xlabel('X1'), plt.ylabel('X2')

#### Optional Plotting

In [ ]:
# Plot the Xtrain points in 2D space
if Xtrain.shape[0] >= 2:
    plt.figure(figsize=(4,4))
    for i in range(Xtrain.shape[1]):
        plt.plot(Xtrain[0,i], Xtrain[1,i], linestyle='None', marker='x', color='red', alpha=0.5)
    plt.xlabel('X1'), plt.ylabel('X2'), plt.grid()
    plt.title('Inducing Points (Xtrain)')
    plt.show()

#### Verify Cost Function Works

In [ ]:
cost1 = get_cost_AC(Z, X, Xplus, Xtrain, ObsManager, nT=nTrain, lambda1=1., lambda2=0., lambda3=0.)
cost2 = get_cost_AC(Z, X, Xplus, Xtrain, ObsManager, nT=nTrain, lambda1=0., lambda2=1., lambda3=0.)
cost3 = get_cost_AC(Z, X, Xplus, Xtrain, ObsManager, nT=nTrain, lambda1=0., lambda2=0., lambda3=1.)
print(f'Initial Cost-1 [NLPD] is {cost1.cpu().detach()}')
print(f'Initial Cost-2 [Linearity] is {cost2.cpu().detach()}')
print(f'Initial Cost-3 [PEM] is {cost3.cpu().detach()}')

### Main Optimization Loop

In [ ]:
# Optimization Parameters: Maximum Iterations | Learning Rate | Target Error
max_iter, learn_rate, err_thresh = 5, 0.001, 0.1

lambda1, lambda2, lambda3 = 10., 10., 0.
# No-Improvement stopping criterion: Iters to monitor | Min decrease in cost
patience, min_delta = 10, 5e-1

# optimizer = torch.optim.Adam([Z] + list(ObsManager.parameters(get_mu_only=True)), lr=learn_rate)

print('Starting Iteration Loop!')
cost_history, iter, count_insignificant = [], 0, 0

# Optimization Options: "Spaced HPOpt" | "Opt all" | "Z only"
routine_name = "Z only"

if routine_name == "Opt all":
    # Gather parameters from each observable in the manager.
    all_hp1, all_hp2, all_noise, all_mu = [], [], [], []
    for obs in ObsManager.observables.values():
        all_hp1.extend(list(obs.hp1_list))
        all_hp2.extend(list(obs.hp2_list))
        all_noise.append(obs.noise)
        all_mu.extend(list(obs.mu_list))
    
    optimizer = torch.optim.Adam([Z] + all_hp1 + all_hp2 + all_noise + all_mu, lr=learn_rate)
    # optimizer = torch.optim.SGD([Z] + all_hp1 + all_hp2, lr=learn_rate, momentum=0.9, nesterov=True)
    # softplus on Kernel parameters is handled inside GPO forward call - modify the package to toggle that option
    while iter < max_iter:
        # Reset gradients to zero
        optimizer.zero_grad()
        
        cost = get_cost_AC(Z, X, Xplus, Xtrain, ObsManager, nT=nTrain, lambda1=lambda1, lambda2=lambda2, lambda3=lambda3)   # compute cost
        cost_history.append(cost.item())    # add to cost history
        cost.backward(retain_graph=False)    # backpropagate

        optimizer.step()    # gradient descent step

        print(f"Iteration {iter}/{max_iter} with {count_insignificant} Insignificant Iterations")
        print(f"Cost: {cost.item()}")

        ObsManager.print_parameters()

        # Stopping conditions
        if cost.item() < err_thresh:
            print("Stopping: Error threshold reached.")
            break
        # Increment iteration
        iter += 1

elif routine_name == "Spaced HPOpt":
    optimizer = torch.optim.Adam([Z], lr=learn_rate)
    while iter < max_iter:
        optimizer.zero_grad()  # Clear gradients
        cost = get_cost_AC(Z, X, Xplus, Xtrain, ObsManager, nT=nTrain, lambda1=0., lambda2=100., lambda3=100.)   # compute cost
        cost_history.append(cost.item())    # add to cost history
        cost.backward(retain_graph=True)    # backpropagate
        optimizer.step()    # gradient descent step
        print(f"Iteration {iter + 1}/{max_iter} with {count_insignificant} Insignificant Iterations")
        print(f"Cost: {cost.item()}")
        if iter % 100 == 0 and iter != 0 and iter < (max_iter-100):
            ObsManager.optimize_hyperparameters(max_iter=50, lr=0.002, opt_mu=False)
            ObsManager.print_parameters()
        # Stopping conditions
        if cost.item() < err_thresh:
            print("Stopping: Error threshold reached.")
            break
        # Increment iteration
        iter += 1

elif routine_name == "Z only":
    # Adam - Non-Stochastic | Faster Convergence, lesser exploration
    # optimizer = torch.optim.Adam([Z], lr=learn_rate)
    # SGD - lr in range 1e-3 | More Exploration.
    optimizer = torch.optim.SGD([Z], lr=learn_rate, momentum=0.9, nesterov=True)
    while iter < max_iter:
        optimizer.zero_grad()  # Clear gradients
        cost = get_cost_AC(Z, X, Xplus, Xtrain, ObsManager, nT=nTrain, lambda1=lambda1, lambda2=lambda2, lambda3=lambda3)   # compute cost
        cost_history.append(cost.item())    # add to cost history
        cost.backward()    # backpropagate
        optimizer.step()    # gradient descent step
        print(f"Iteration {iter + 1}/{max_iter} with {count_insignificant} Insignificant Iterations")
        print(f"Cost: {cost.item()}")
        # Stopping conditions
        if cost.item() < err_thresh:
            print("Stopping: Error threshold reached.")
            break
        # Increment iteration
        iter += 1

elif routine_name == "Z and hp-opt":
    # Gather parameters from each observable in the manager.
    all_hp1, all_hp2, all_noise = [], [], []
    for obs in ObsManager.observables.values():
        all_hp1.extend(list(obs.hp1_list))
        all_hp2.extend(list(obs.hp2_list))
        # all_noise.append(obs.noise)
    
    optimizer = torch.optim.Adam([Z] + all_hp1 + all_hp2, lr=learn_rate)
    # optimizer = torch.optim.SGD([Z] + all_hp1 + all_hp2, lr=learn_rate, momentum=0.9, nesterov=True)
    while iter < max_iter:
        # Reset gradients to zero
        optimizer.zero_grad()
        
        cost = get_cost_AC(Z, X, Xplus, Xtrain, ObsManager, nT=nTrain, lambda1=lambda1, lambda2=lambda2, lambda3=lambda3)   # compute cost
        cost_history.append(cost.item())    # add to cost history
        cost.backward(retain_graph=False)    # backpropagate
        for i, theta in enumerate(all_hp1):
            if theta.grad is not None:
                theta.grad.data = torch.nan_to_num(theta.grad.data)
                theta.grad.data = torch.clamp(theta.grad.data, min=-1e2, max=1e2)
                print(f"hp1 parameter {i} grad: {theta.grad}")
                
            else:
                print(f"hp1 parameter {i} has no gradient.")
        # Print gradients for hp2_list parameters
        print("Gradients for hp2:")
        for i, theta in enumerate(all_hp2):
            if theta.grad is not None:
                theta.grad.data = torch.nan_to_num(theta.grad.data)
                theta.grad.data = torch.clamp(theta.grad.data, min=-1e2, max=1e2)
                print(f"hp2 parameter {i} grad: {theta.grad}")

            else:
                print(f"hp2 parameter {i} has no gradient.")

        # Uncomment following if optimizing noise too
        # print("Gradients for noise:")
        # for i, theta in enumerate(all_noise):
        #     if theta.grad is not None:
        #         print(f"noise parameter {i} grad: {theta.grad}")
        #     else:
        #         print(f"noise parameter {i} has no gradient.")

        optimizer.step()    # gradient descent step

        print(f"Iteration {iter}/{max_iter} with {count_insignificant} Insignificant Iterations")
        print(f"Cost: {cost.item()}")

        ObsManager.print_parameters()

        # Stopping conditions
        if cost.item() < err_thresh:
            print("Stopping: Error threshold reached.")
            break
        # Increment iteration
        iter += 1

else:
    raise ValueError('Invalid routine_name')

if iter == max_iter:    # Print Stopping Criterion
    print(f'Stopping: Reached maximum number of iterations = {iter}.')

optimal_Z = Z.detach()
print('Optimization Complete.')
print("Final Cost:", cost.item())

### View Optimization Logs

In [ ]:
# Cost Plotting
# Plot cost history
plt.figure(figsize=(6,4))
plt.plot(cost_history, label="Cost")
plt.title("Cost History")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.legend(), plt.grid()
plt.show()

# Logarithmic cost history
plt.figure(figsize=(6,4))
plt.plot(torch.log10(torch.tensor(cost_history)), label="log(Cost)")
plt.title("Logarithmic Cost History")
plt.xlabel("Iteration")
plt.ylabel("log(Cost)")
plt.legend(), plt.grid()
plt.show()

In [ ]:
ObsManager.print_parameters()

### Post-Process Optimization Results

In [ ]:
p=10
# Use Optimal Z values to Build GP Models and Optimal A and Z matrices
optimal_Z = Z.detach()
for i in range(p):
    ObsManager.train_observable(i, Xtrain, optimal_Z[:,i])  # train GP Observables with Optimal Z outputs

# ObsManager.optimize_hyperparameters(opt_mu=False, opt_sigma=True, max_iter=25)   # Optimize Kernel hyperparameters for Optimal training data
# print(f'GPO Hyperparameters have been optimized.')
ObsManager.print_parameters()

ObsList = [i for i in range(10)]
A, C = gpk.getKoopman(ObsManager, ObsList, Xall, nTrain, stateAug=False)

# Simulation and Validation

## Model Simulation

In [ ]:
# Evaluation on training set
ZmeanTrain = torch.empty((nTrain, p, N))    # n+p for state-augmentation
ZcvTrain = torch.empty((nTrain, p, p, N))   # n for no state-augmentation
#ZmeanTrain[:, :n, 0] = ICsetTrain.T    # only for state-augmentation

XhatTrain = torch.empty((nTrain, n, N))
XcvhatTrain = torch.empty((nTrain, n, n, N))
TrainRMSE = torch.empty((nTrain,n))

for j in range(nTrain): # GP Predict for all training trajectories
    for i in range(p):  # GP predict IC and IC-cv
        ZmeanTrain[j, i, 0] = ObsManager.predict_mean(i, ICsetTrain[:, j].view(n,1))        # n+i for state-augmentation
        ZcvTrain[j, i, i, 0] = ObsManager.predict_covariance(i, ICsetTrain[:, j].view(n,1)) # n for no state-augmentation
    
    ZmeanTrain[j, :, :], ZcvTrain[j, :, :, :], XhatTrain[j, :, :], XcvhatTrain[j, :, :, :] = gpk.sim_LTI(ZmeanTrain[j,:,0], A, C, num_steps=N, ts=None, x0cv=ZcvTrain[j,:,:,0])
    TrainRMSE[j,:] = torch.sqrt(torch.mean((XhatTrain[j,:,:] - SimData[j,:,:N])**2,1))

# Evaluation on test set
ZmeanTest = torch.empty((nTest, p, N))
ZcvTest = torch.empty((nTest, p, p, N))
#ZmeanTest[:, :n, 0] = ICsetTest.T  # only for state-augmentation

XhatTest = torch.empty((nTest, n, N))
XcvhatTest = torch.empty((nTest, n, n, N))
TestRMSE = torch.empty((nTest,n))

for j in range(nTest): # GP Predict for all testing trajectories
    for i in range(p):  # GP predict IC and IC-cv
        ZmeanTest[j, i, 0] = ObsManager.predict_mean(i, ICsetTest[:, j].view(n,1))          # n+i for state-augmentation
        ZcvTest[j, i, i, 0] = ObsManager.predict_covariance(i, ICsetTest[:, j].view(n,1))   # n for no state-augmentation

    ZmeanTest[j, :, :], ZcvTest[j, :, :, :], XhatTest[j, :, :], XcvhatTest[j, :, :, :] = gpk.sim_LTI(ZmeanTest[j,:,0], A, C, num_steps=N, ts=None, x0cv=ZcvTest[j,:,:,0])
    TestRMSE[j,:] = torch.sqrt(torch.mean((XhatTest[j,:,:] - SimData[nTest+j,:,:N])**2,1))

XhatTrain, XhatTest, XcvhatTrain, XcvhatTest = XhatTrain.detach(), XhatTest.detach(), XcvhatTrain.detach(), XcvhatTest.detach()
TestRMSE, TrainRMSE = TestRMSE.detach(), TrainRMSE.detach()

In [ ]:
# Function to compute RMSE, NLPD, and sRMSE
def compute_metrics(Xhat, Xcvhat, SimData, nTraj, N, eps=1e-6):
    RMSE = torch.sqrt(torch.mean((Xhat - SimData[:nTraj, :, :N]) ** 2, dim=2))
    
    var_diag = Xcvhat.diagonal(dim1=1, dim2=2).permute(0, 2, 1)  # Shape (nTraj, n, N)
    
    NLPD = torch.mean(0.5 * ((SimData[:nTraj, :, :N] - Xhat) ** 2 / var_diag + torch.log(2 * torch.pi * var_diag)), dim=2)
    
    #sRMSE = torch.sqrt(torch.mean(((SimData[:nTraj, :, :N] - Xhat) / var_diag.sqrt()) ** 2, dim=2))

    return RMSE, NLPD

time = torch.arange(0., ts * N, ts)
idx1, idx2, idx3 = torch.argmin(TrainRMSE.mean(dim=1)), torch.argmin(TestRMSE.mean(dim=1)), torch.argmax(TestRMSE.mean(dim=1))

## Trajectory and Time-Evolution Plots

In [ ]:
gpk.plot_phase(XhatTrain, SimData, ICsetTrain, idx1, N, system_name, 'Training Trajectory')
gpk.plot_time_series_with_bounds(time, XhatTrain, XcvhatTrain, SimData, idx1, N, system_name, title_suffix='Best Train Trajectory')

gpk.plot_phase(XhatTest, SimData, ICsetTest, idx2, N, system_name, 'Best Test Trajectory', sim_offset=nTrain)
gpk.plot_time_series_with_bounds(time, XhatTest, XcvhatTest, SimData, idx2, N, system_name, title_suffix='Best Test Trajectory', sim_offset=nTrain)

gpk.plot_phase(XhatTest, SimData, ICsetTest, idx3, N, system_name, 'Worst Test Trajectory', sim_offset=nTrain)
gpk.plot_time_series_with_bounds(time, XhatTest, XcvhatTest, SimData, idx3, N, system_name, title_suffix='Worst Test Trajectory', sim_offset=nTrain)


In [ ]:
# For best test trajectory (e.g., idx2)
gpk.plot_predicted_sd_error(XcvhatTest, SimData, XhatTest, idx=idx2, N=N, nTrain=nTrain, trajectory_label="Best Test")

# For worst test trajectory (e.g., idx3)
gpk.plot_predicted_sd_error(XcvhatTest, SimData, XhatTest, idx=idx3, N=N, nTrain=nTrain, trajectory_label="Worst Test")

## System-level Plots

In [ ]:
# Plot Errors for all trajectories
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Training set plot
axes[0].plot(range(nTrain), TrainRMSE[:,0].numpy(), marker='o', linestyle='-', label='RMSE X1')
axes[0].plot(range(nTrain), TrainRMSE[:,1].numpy(), marker='o', linestyle='-', label='RMSE X2')
#axes[0].plot(range(nTrain), TrainNLPD.mean(dim=1).numpy(), marker='s', linestyle='-', label='NLPD')
#axes[0].plot(range(nTrain), TrainsRMSE.mean(dim=1).numpy(), marker='^', linestyle='-', label='sRMSE')
axes[0].set_title('Training Metrics')
axes[0].set_xlabel("Trajectory Index")
axes[0].set_ylabel("Metric Value")
axes[0].legend()
axes[0].grid()

# Test set plot
axes[1].plot(range(nTest), TestRMSE[:,0].numpy(), marker='o', linestyle='-', label='RMSE X1')
axes[1].plot(range(nTest), TestRMSE[:,1].numpy(), marker='o', linestyle='-', label='RMSE X2')
#axes[1].plot(range(nTest), TestNLPD.mean(dim=1).numpy(), marker='s', linestyle='-', label='NLPD')
#axes[1].plot(range(nTest), TestsRMSE.mean(dim=1).numpy(), marker='^', linestyle='-', label='sRMSE')
axes[1].set_title('Test Metrics')
axes[1].set_xlabel("Trajectory Index")
axes[1].set_ylabel("Metric Value")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
# Eigen value plot of Koopman Matrices
eigval = torch.linalg.eigvals(A)

eigreal, eigimag = eigval.real, eigval.imag
eigreal, eigimag = eigreal.detach().numpy(), eigimag.detach().numpy()

theta = np.linspace(0, 2*np.pi, 500)
unitCirclex, unitCircley = np.cos(theta), np.sin(theta)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
# First subplot: Eigenvalues plot
axes[0].plot(unitCirclex, unitCircley, color='red', label='Unit Circle')
axes[0].scatter(eigreal, eigimag, color='blue', label='Eigenvalues')
axes[0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_title(f"Eigenvalues of A Matrix with {p} Observables")
axes[0].set_xlabel("Real Part")
axes[0].set_ylabel("Imaginary Part")
axes[0].grid(True)
axes[0].legend(loc='upper right')

# Second subplot: Heatmap of matrix A
im = axes[1].imshow(A.detach().numpy(), cmap='viridis', aspect='auto')
fig.colorbar(im, ax=axes[1], label="Value")
axes[1].set_title(f'{A.shape[0]}-D Koopman Matrix')
axes[1].set_xlabel("Columns")
axes[1].set_ylabel("Rows")
plt.tight_layout()
plt.show()

In [ ]:
# Phase Diagram from all IC simulation
if SimData.shape[1] == 3:       # 3D plot
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    for j in range(SimData.shape[0]):
        ax.plot(SimData[j, 0, :], SimData[j, 1, :], SimData[j, 2, :],
                alpha=0.5, color='blue')
        ax.scatter(SimData[j, 0, 0], SimData[j, 1, 0], SimData[j, 2, 0],
                   color='red')
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')
    ax.set_zlabel('X3')
    ax.set_title(f'All Trajectories: {system_name}')
    ax.grid(True)
    plt.show()

elif SimData.shape[1] == 2:     # 2D plot
    for j in range(SimData.shape[0]):
        plt.plot(SimData[j, 0, :], SimData[j, 1, :],
                 alpha=0.5, color='blue', linewidth=1.)
        plt.plot(SimData[j, 0, 0], SimData[j, 1, 0],
                 'o', color='red')
    plt.grid()
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(f'All Trajectories: {system_name}')
    plt.show()

elif SimData.shape[1] == 1:     # 1D plot
    for j in range(SimData.shape[0]):
        plt.plot(SimData[j, 0, :], alpha=0.5, color='blue', linewidth=1.)
        plt.plot(SimData[j, 0, 0], 'o', color='red')
    plt.grid()
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(f'All Trajectories: {system_name}')
    plt.show()

else:
    print('IC Phase Plots supported for 1/2/3-D systems only.')


# Training Set Predicted Trajectories
if XhatTrain.shape[1] == 3: # 3D plot
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    for j in range(XhatTrain.shape[0]):
        ax.plot(XhatTrain[j, 0, :], XhatTrain[j, 1, :], XhatTrain[j, 2, :],
                alpha=0.5, color='blue')
        ax.scatter(XhatTrain[j, 0, 0], XhatTrain[j, 1, 0], XhatTrain[j, 2, 0],
                   color='red')
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')
    ax.set_zlabel('X3')
    ax.set_title(f'Predicted Trajectories - Training Set: {system_name}')
    ax.grid(True)
    plt.show()

elif XhatTrain.shape[1] == 2:   # 2D plot
    for j in range(XhatTrain.shape[0]):
        plt.plot(XhatTrain[j, 0, :], XhatTrain[j, 1, :],
                 alpha=0.5, color='blue', linewidth=1.)
        plt.plot(XhatTrain[j, 0, 0], XhatTrain[j, 1, 0],
                 'o', color='red')
    plt.grid()
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(f'Predicted Trajectories - Training Set: {system_name}')
    plt.show()

else:
    print('Trianing Response Phase PLots supported for 2/3-D systems only.')

# Model Saving

In [ ]:
today = datetime.date.today()

iGPKResults = {
    "Sysem Name": system_name,
    "Data": SimData,
    "Train Test Factors": [nTrain, nTest],
    "Initial Conditions": [ICsetTrain, ICsetTest],
    "ObsManager": ObsManager,
    "Observables": ObsManager.observables,
    "Koopman A": A,
    "Koopman C": C,
    "X Train": Xtrain,
    "Optimal Z": optimal_Z,
    "Train RMSE": TrainRMSE,
    "Test RMSE": TestRMSE,
    "Adam Options": [learn_rate, max_iter, lambda1, lambda2, lambda3]
}

torch.save(iGPKResults, f'Results/iGPKAuto_{system_name}_Noisy_{today}.pt')